In [18]:
import pandas as pd
from tmdbv3api import TMDb, Movie
from dotenv import load_dotenv

In [19]:
df_links = pd.read_csv('../data/links.csv')
df_movies = pd.read_csv('../data/movies.csv')
df_ratings = pd.read_csv('../data/ratings.csv')
df_tags = pd.read_csv('../data/tags.csv')
print(len(df_ratings))
print(len(df_links))
print(len(df_movies))

100836
9742
9742


In [20]:
def head(df):
    return df.head()

def isnull(df):
    return df.isnull().sum()

print(head(df_links))
print(head(df_movies))
print(head(df_ratings))
head(df_tags)

   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [21]:
print(isnull(df_links))
print(isnull(df_movies))
print(isnull(df_ratings))
print(isnull(df_tags))

movieId    0
imdbId     0
tmdbId     8
dtype: int64
movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64


In [22]:
#pip install tmdbv3api
load_dotenv()   

True

In [23]:
tmdb = TMDb()
tmdb.api_key = os.getenv("TMDB_API_KEY")
tmdb.language = 'fr'
movie_api = Movie()
enriched_data = []

first_valid_row = df_links.dropna(subset=['tmdbId']).iloc[0]
tmdb_id = int(first_valid_row['tmdbId'])
m = movie_api.details(int(tmdb_id))
print(m.__dict__.keys())

dict_keys(['_json', '_key', '_dict_key', '_dict_key_name', '_obj_list', '_list_only', 'adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'videos', 'trailers', 'images', 'casts', 'translations', 'keywords', 'release_dates'])


<p style = 'color:blue;font-weight:bold'>
I/ Data Enrichement
<p>

on enrichit les données avec l'API TMDb pour avoir du contexte. 'links.csv' contient seulement les caractéristiques 'movieId','imdbId','tmdbId'

In [24]:
"""" 
for index, row in tqdm(df_links.iterrows(), total = df_links.shape[0]):
    tmdb_id = row['tmdbId']
    
    if pd.isna(tmdb_id):
        enriched_data.append({})
        continue
    
    try:
        m = movie_api.details(int(tmdb_id))
        enriched_data.append({
            'movieId': row['movieId'], 
            'budget': getattr(m, 'budget', 0),
            'revenue': getattr(m, 'revenue', 0),
            'runtime': getattr(m, 'runtime', 0),
            'release_date': getattr(m, 'release_date', np.nan),
            'vote_average_tmdb': getattr(m, 'vote_average', 0),
            'vote_count_tmdb': getattr(m, 'vote_count', 0)
        })
        
    except Exception as e:
        enriched_data.append({'movieId': row['movieId']})
        
df_enriched = pd.DataFrame(enriched_data)
df_enriched.to_csv('../data/movies_enriched.csv', index = False)
df_meta = pd.read_csv('../data/movies_enriched.csv')
"""

'" \nfor index, row in tqdm(df_links.iterrows(), total = df_links.shape[0]):\n    tmdb_id = row[\'tmdbId\']\n\n    if pd.isna(tmdb_id):\n        enriched_data.append({})\n        continue\n\n    try:\n        m = movie_api.details(int(tmdb_id))\n        enriched_data.append({\n            \'movieId\': row[\'movieId\'], \n            \'budget\': getattr(m, \'budget\', 0),\n            \'revenue\': getattr(m, \'revenue\', 0),\n            \'runtime\': getattr(m, \'runtime\', 0),\n            \'release_date\': getattr(m, \'release_date\', np.nan),\n            \'vote_average_tmdb\': getattr(m, \'vote_average\', 0),\n            \'vote_count_tmdb\': getattr(m, \'vote_count\', 0)\n        })\n\n    except Exception as e:\n        enriched_data.append({\'movieId\': row[\'movieId\']})\n\ndf_enriched = pd.DataFrame(enriched_data)\ndf_enriched.to_csv(\'../data/movies_enriched.csv\', index = False)\ndf_meta = pd.read_csv(\'../data/movies_enriched.csv\')\n'

In [25]:
df_meta = pd.read_csv('../data/movies_enriched.csv')
df_meta.columns


Index(['movieId', 'budget', 'revenue', 'runtime', 'release_date',
       'vote_average_tmdb', 'vote_count_tmdb'],
      dtype='str')

In [26]:
df_ratings_sumup = df_ratings.groupby('movieId').agg({'rating' : ['mean', 'count', 'std']})
df_ratings_sumup = df_ratings_sumup.reset_index()
df_ratings_sumup.columns = ['movieId', 'mean_rating', 'num_votes', 'std_rating']


In [27]:
df_ratings_sumup.head()

,movieId,mean_rating,num_votes,std_rating
0,1,3.920930,215,0.834859
1,2,3.431818,110,0.881713
2,3,3.259615,52,1.054823
3,4,2.357143,7,0.852168
4,5,3.071429,49,0.907148


In [28]:
len(df_ratings_sumup)

9724

In [29]:
df_tags['tag'] = df_tags['tag'].astype(str)
df_tags_sumup = df_tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
df_tags_sumup.columns = ['movieId', 'all_tags']

<p style = 'color:blue; font-weight: bold'>
Merging Datasets
<p>

In [30]:
df_final = df_movies.merge(df_meta, on = 'movieId', how = 'left')
df_final = df_final.merge(df_ratings_sumup, on = 'movieId', how = 'left')
df_final = df_final.merge(df_tags_sumup,on = 'movieId', how = 'left')
print(df_final.columns)
print(df_final.head())
print(len(df_final))

Index(['movieId', 'title', 'genres', 'budget', 'revenue', 'runtime',
       'release_date', 'vote_average_tmdb', 'vote_count_tmdb', 'mean_rating',
       'num_votes', 'std_rating', 'all_tags'],
      dtype='str')
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres      budget      revenue  \
0  Adventure|Animation|Children|Comedy|Fantasy  30000000.0  401157969.0   
1                   Adventure|Children|Fantasy  65000000.0  262821940.0   
2                               Comedy|Romance  25000000.0   71500000.0   
3                         Comedy|Drama|Romance  16000000.0   81452156.0   
4                                       Comedy         0.0   76594107.0   

   runtime release_date  vote_avera

In [31]:
df_final.to_csv('../data/df_final.csv', index = True)